# Resumo dinâmico de avaliações com GroqCloud

In [1]:
!pip -q install groq openpyxl ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.1 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import time
import math
import pandas as pd
import ipywidgets as widgets
from getpass import getpass
from IPython.display import display, Markdown, clear_output
from groq import Groq

MODEL = "llama-3.3-70b-versatile"

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Cole sua GROQ_API_KEY: ")

client = Groq(api_key=GROQ_API_KEY)
print("Cliente Groq configurado.")

Cole sua GROQ_API_KEY: ··········
Cliente Groq configurado.


In [4]:
from google.colab import files
uploaded = files.upload()
ARQUIVO = next(iter(uploaded))

df = pd.read_excel(ARQUIVO)
df.columns = [str(c).strip() for c in df.columns]

print("Linhas:", len(df))
print("Colunas:", list(df.columns))
display(df.head())

Saving planilha_avaliacoes_whey.xlsx to planilha_avaliacoes_whey.xlsx
Linhas: 62
Colunas: ['Produto', 'Usuário', 'Nota', 'Avaliação', 'Data', 'Sabor', 'Compra verificada', 'Descrição', 'Úteis', 'Sentimento', 'Tema principal', 'Ponto de atenção']


,Produto,Usuário,Nota,Avaliação,Data,Sabor,Compra verificada,Descrição,Úteis,Sentimento,Tema principal,Ponto de atenção
0,Whey Protein - Doce de Leite,Amanda Cruz,5,Nuuuu é bom demais!,2025-05-07,Doce de Leite,Sim,Sou muito suspeita pra falar pois amo doce de ...,2,Positivo,"Sabor, Custo-benefício",NaN
1,Whey Protein - Doce de Leite,Anônima,5,Whey aprovadíssimo!!,2025-08-12,Doce de Leite,Sim,"Produto muito bom, valeu super a pena. Não sen...",3,Positivo,Sabor,NaN
2,Whey Protein - Doce de Leite,Ley,4,Gosto enjoativo,2025-09-30,Doce de Leite,Sim,"Ele entrega o que promete, só o gosto que é en...",1,Positivo com ressalva,"Sabor, Ponto de atenção",sabor enjoativo
3,Whey Protein - Doce de Leite,Jaiane,5,Whey,2026-02-06,Doce de Leite,Sim,Ótimo sabor,0,Positivo,Sabor,NaN
4,Whey Protein - Doce de Leite,Maria Vieira,5,Proteína de boa qualidade é fundamental.,2026-01-27,Doce de Leite,Sim,Proteína de excelente qualidade com bom preço.,0,Positivo,"Custo-benefício, Qualidade/Composição",NaN


In [5]:
def primeira_coluna_existente(dados, opcoes):
    for col in opcoes:
        if col in dados.columns:
            return col
    return None

COL_TITULO = primeira_coluna_existente(df, ["Avaliação", "Titulo", "Título", "Review", "Comentario", "Comentário"])
COL_DESC = primeira_coluna_existente(df, ["Descrição", "Descricao", "Comentário", "Comentario", "Texto", "Review Text"])
COL_NOTA = primeira_coluna_existente(df, ["Nota", "Estrelas", "Rating"])
COL_SABOR = primeira_coluna_existente(df, ["Sabor", "Variação", "Variacao"])
COL_SENT = primeira_coluna_existente(df, ["Sentimento", "Sentiment"])
COL_TEMA = primeira_coluna_existente(df, ["Tema principal", "Tema", "Categoria"])
COL_ATENCAO = primeira_coluna_existente(df, ["Ponto de atenção", "Ponto de atencao", "Problema", "Atenção"])

if COL_DESC is None and COL_TITULO is None:
    raise ValueError("Não encontrei coluna de texto. Verifique se existe uma coluna como 'Descrição' ou 'Avaliação'.")

def texto_linha(row):
    partes = []
    if COL_TITULO and pd.notna(row.get(COL_TITULO)):
        partes.append(str(row.get(COL_TITULO)).strip())
    if COL_DESC and pd.notna(row.get(COL_DESC)):
        desc = str(row.get(COL_DESC)).strip()
        if desc and desc not in partes:
            partes.append(desc)
    return " - ".join([p for p in partes if p])

df["texto_comentario"] = df.apply(texto_linha, axis=1)
df = df[df["texto_comentario"].str.len() > 0].copy()

if "Data" in df.columns and pd.api.types.is_numeric_dtype(df["Data"]):
    df["Data_convertida"] = pd.to_datetime(df["Data"], unit="D", origin="1899-12-30", errors="coerce")

print("Comentários válidos:", len(df))
display(df[[c for c in [COL_NOTA, COL_SABOR, COL_SENT, COL_TEMA, COL_ATENCAO, "texto_comentario"] if c]].head(10))

Comentários válidos: 62


,Nota,Sabor,Sentimento,Tema principal,Ponto de atenção,texto_comentario
0,5,Doce de Leite,Positivo,"Sabor, Custo-benefício",NaN,Nuuuu é bom demais! - Sou muito suspeita pra f...
1,5,Doce de Leite,Positivo,Sabor,NaN,"Whey aprovadíssimo!! - Produto muito bom, vale..."
2,4,Doce de Leite,Positivo com ressalva,"Sabor, Ponto de atenção",sabor enjoativo,"Gosto enjoativo - Ele entrega o que promete, s..."
3,5,Doce de Leite,Positivo,Sabor,NaN,Whey - Ótimo sabor
4,5,Doce de Leite,Positivo,"Custo-benefício, Qualidade/Composição",NaN,Proteína de boa qualidade é fundamental. - Pro...
5,3,Doce de Leite,Neutro / Atenção,"Sabor, Textura/Diluição, Entrega/Embalagem",lacre violado,Não sei se é original - Coloquei 3 estrelas po...
6,5,Doce de Leite,Positivo,Comentário geral,NaN,Otimo - Muito bom!
7,5,Doce de Leite,Positivo,"Custo-benefício, Qualidade/Composição",NaN,FTW nova Growth! - Ftw sempre a melhor na ques...
8,5,Doce de Leite,Positivo,Sabor,NaN,Delícia! - Delicioso! Rende muito.
9,5,Doce de Leite,Positivo,Comentário geral,NaN,Whey - exceçlente


In [6]:
def extrair_json(texto):
    """Tenta transformar a resposta do modelo em dicionário JSON."""
    if not texto:
        return {"erro": "Resposta vazia do modelo."}
    texto = texto.strip()
    try:
        return json.loads(texto)
    except Exception:
        pass
    match = re.search(r"\{.*\}", texto, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    return {"resposta_bruta": texto}


def chamar_groq(prompt_usuario, temperature=0.2, max_completion_tokens=1400):
    """Chama o modelo com tratamento simples de erro e retry."""
    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um analista de avaliações de e-commerce. "
                "Responda sempre em português brasileiro. "
                "Quando pedirem JSON, responda somente com JSON válido, sem markdown."
            ),
        },
        {"role": "user", "content": prompt_usuario},
    ]

    ultimo_erro = None
    for tentativa in range(3):
        try:
            resposta = client.chat.completions.create(
                model=MODEL,
                messages=mensagens,
                temperature=temperature,
                max_completion_tokens=max_completion_tokens,
            )
            return resposta.choices[0].message.content
        except Exception as e:
            ultimo_erro = e
            time.sleep(1.5 * (tentativa + 1))
    raise RuntimeError(f"Erro ao chamar a Groq depois de 3 tentativas: {ultimo_erro}")


def formatar_avaliacoes(dados, limite=None):
    """Transforma as linhas filtradas em texto compacto para o LLM."""
    if limite:
        dados = dados.head(limite)

    linhas = []
    for idx, row in dados.iterrows():
        meta = []
        if COL_NOTA and pd.notna(row.get(COL_NOTA)):
            meta.append(f"Nota={row.get(COL_NOTA)}")
        if COL_SABOR and pd.notna(row.get(COL_SABOR)):
            meta.append(f"Sabor={row.get(COL_SABOR)}")
        if COL_SENT and pd.notna(row.get(COL_SENT)):
            meta.append(f"Sentimento={row.get(COL_SENT)}")
        if COL_TEMA and pd.notna(row.get(COL_TEMA)):
            meta.append(f"Tema={row.get(COL_TEMA)}")
        if COL_ATENCAO and pd.notna(row.get(COL_ATENCAO)):
            meta.append(f"Atenção={row.get(COL_ATENCAO)}")

        comentario = str(row["texto_comentario"]).replace("\n", " ").strip()
        linhas.append(f"[{idx}] {'; '.join(meta)} | Comentário: {comentario}")

    return "\n".join(linhas)


def prompt_resumo(avaliacoes_texto, total_linhas, foco="geral"):
    return f"""
Você receberá avaliações de consumidores sobre um produto.

Objetivo do resumo: {foco}
Total de avaliações consideradas neste lote: {total_linhas}

Gere um JSON válido exatamente neste formato:
{{
  "resumo_geral": "texto com até 120 palavras",
  "principais_pontos_positivos": ["ponto 1", "ponto 2", "ponto 3"],
  "principais_reclamacoes": ["reclamação 1", "reclamação 2", "reclamação 3"],
  "pontos_de_atencao": ["atenção 1", "atenção 2"],
  "temas_mais_citados": [
    {{"tema": "tema", "explicacao": "explicação curta"}}
  ],
  "perfil_do_cliente": "descrição curta do tipo de cliente que tende a gostar ou não gostar",
  "frase_executiva": "uma frase objetiva para apresentação",
  "confianca": "alta, média ou baixa"
}}

Regras:
- Não invente fatos que não aparecem nos comentários.
- Diferencie elogios frequentes de reclamações pontuais.
- Se houver poucos comentários, deixe isso claro no campo de confiança.
- Não cite nomes de usuários.
- Responda somente o JSON.

Avaliações:
{avaliacoes_texto}
""".strip()


def dividir_dataframe(dados, tamanho_lote=45):
    for inicio in range(0, len(dados), tamanho_lote):
        yield dados.iloc[inicio:inicio + tamanho_lote]


def resumir_dataframe(dados, foco="geral", tamanho_lote=45):
    """
    Resume um dataframe filtrado.
    Se houver muitos comentários, resume por lotes e depois consolida os resumos.
    """
    dados = dados.copy()
    total = len(dados)
    if total == 0:
        return {"erro": "Nenhum comentário encontrado com esses filtros."}

    if total <= tamanho_lote:
        texto = formatar_avaliacoes(dados)
        bruto = chamar_groq(prompt_resumo(texto, total, foco=foco))
        return extrair_json(bruto)

    resumos_lote = []
    for n, lote in enumerate(dividir_dataframe(dados, tamanho_lote=tamanho_lote), start=1):
        texto = formatar_avaliacoes(lote)
        bruto = chamar_groq(prompt_resumo(texto, len(lote), foco=f"{foco} | lote {n}"))
        resumos_lote.append(extrair_json(bruto))
        time.sleep(0.4)

    prompt_final = f"""
Consolide os resumos parciais abaixo em um único resumo final.
Total real de avaliações analisadas: {total}
Objetivo: {foco}

Resumos parciais em JSON:
{json.dumps(resumos_lote, ensure_ascii=False, indent=2)}

Gere um JSON válido no mesmo formato:
{{
  "resumo_geral": "texto com até 140 palavras",
  "principais_pontos_positivos": ["ponto 1", "ponto 2", "ponto 3"],
  "principais_reclamacoes": ["reclamação 1", "reclamação 2", "reclamação 3"],
  "pontos_de_atencao": ["atenção 1", "atenção 2"],
  "temas_mais_citados": [
    {{"tema": "tema", "explicacao": "explicação curta"}}
  ],
  "perfil_do_cliente": "descrição curta",
  "frase_executiva": "uma frase objetiva para apresentação",
  "confianca": "alta, média ou baixa"
}}

Responda somente o JSON.
""".strip()
    bruto_final = chamar_groq(prompt_final, max_completion_tokens=1600)
    return extrair_json(bruto_final)

In [7]:
def opcoes_coluna(col):
    if col and col in df.columns:
        valores = sorted(df[col].dropna().astype(str).unique().tolist())
        return ["Todos"] + valores
    return ["Todos"]

sabor_widget = widgets.Dropdown(options=opcoes_coluna(COL_SABOR), description="Sabor:")
sent_widget = widgets.Dropdown(options=opcoes_coluna(COL_SENT), description="Sentimento:")

if COL_NOTA and COL_NOTA in df.columns:
    notas_validas = pd.to_numeric(df[COL_NOTA], errors="coerce").dropna()
    min_nota_val = int(notas_validas.min()) if len(notas_validas) else 1
    max_nota_val = int(notas_validas.max()) if len(notas_validas) else 5
else:
    min_nota_val, max_nota_val = 1, 5

nota_min_widget = widgets.IntSlider(value=min_nota_val, min=min_nota_val, max=max_nota_val, step=1, description="Nota mín.:")
foco_widget = widgets.Dropdown(
    options=[
        "geral",
        "sabor e aceitação do produto",
        "reclamações e pontos de atenção",
        "custo-benefício",
        "qualidade/composição",
        "entrega, embalagem e lacre",
        "resumo para apresentação acadêmica",
    ],
    value="geral",
    description="Foco:",
)

botao = widgets.Button(description="Gerar resumo", button_style="success")
botao_exportar = widgets.Button(description="Exportar histórico", button_style="info")
saida = widgets.Output()
historico_resumos = []


def aplicar_filtros():
    dados = df.copy()
    if COL_SABOR and sabor_widget.value != "Todos":
        dados = dados[dados[COL_SABOR].astype(str) == sabor_widget.value]
    if COL_SENT and sent_widget.value != "Todos":
        dados = dados[dados[COL_SENT].astype(str) == sent_widget.value]
    if COL_NOTA and COL_NOTA in dados.columns:
        dados[COL_NOTA] = pd.to_numeric(dados[COL_NOTA], errors="coerce")
        dados = dados[dados[COL_NOTA] >= nota_min_widget.value]
    return dados


def lista_markdown(titulo, itens):
    if not itens:
        return f"\n**{titulo}:** não identificado.\n"
    if isinstance(itens, str):
        itens = [itens]
    texto = f"\n**{titulo}:**\n"
    for item in itens:
        if isinstance(item, dict):
            tema = item.get("tema", "Tema")
            explicacao = item.get("explicacao", "")
            texto += f"- **{tema}:** {explicacao}\n"
        else:
            texto += f"- {item}\n"
    return texto


def renderizar_resultado(resultado, dados):
    if "erro" in resultado:
        return f"### Erro\n{resultado['erro']}"
    if "resposta_bruta" in resultado:
        return "### Resposta do modelo\n" + resultado["resposta_bruta"]

    md = f"### Resumo dinâmico\n"
    md += f"**Comentários analisados:** {len(dados)}\n\n"
    md += f"**Resumo geral:** {resultado.get('resumo_geral', 'Não informado.')}\n"
    md += lista_markdown("Principais pontos positivos", resultado.get("principais_pontos_positivos", []))
    md += lista_markdown("Principais reclamações", resultado.get("principais_reclamacoes", []))
    md += lista_markdown("Pontos de atenção", resultado.get("pontos_de_atencao", []))
    md += lista_markdown("Temas mais citados", resultado.get("temas_mais_citados", []))
    md += f"\n**Perfil do cliente:** {resultado.get('perfil_do_cliente', 'Não informado.')}\n"
    md += f"\n**Frase executiva:** {resultado.get('frase_executiva', 'Não informado.')}\n"
    md += f"\n**Confiança:** {resultado.get('confianca', 'Não informada.')}\n"
    return md


def ao_clicar_gerar(_):
    with saida:
        clear_output()
        dados = aplicar_filtros()
        display(Markdown(f"Gerando resumo para **{len(dados)}** comentários..."))
        resultado = resumir_dataframe(dados, foco=foco_widget.value)
        clear_output()
        display(Markdown(renderizar_resultado(resultado, dados)))

        historico_resumos.append({
            "sabor": sabor_widget.value,
            "sentimento": sent_widget.value,
            "nota_minima": nota_min_widget.value,
            "foco": foco_widget.value,
            "comentarios_analisados": len(dados),
            "resumo_json": json.dumps(resultado, ensure_ascii=False),
            "resumo_geral": resultado.get("resumo_geral", "") if isinstance(resultado, dict) else "",
            "frase_executiva": resultado.get("frase_executiva", "") if isinstance(resultado, dict) else "",
        })


def ao_clicar_exportar(_):
    with saida:
        if not historico_resumos:
            display(Markdown("Nenhum resumo foi gerado ainda."))
            return
        arquivo_saida = "historico_resumos_groq.xlsx"
        pd.DataFrame(historico_resumos).to_excel(arquivo_saida, index=False)
        files.download(arquivo_saida)

botao.on_click(ao_clicar_gerar)
botao_exportar.on_click(ao_clicar_exportar)

display(widgets.VBox([
    widgets.HBox([sabor_widget, sent_widget]),
    widgets.HBox([nota_min_widget, foco_widget]),
    widgets.HBox([botao, botao_exportar]),
    saida,
]))

In [8]:
def resumir_por_grupo(coluna_grupo="Sabor", foco="geral"):
    if coluna_grupo not in df.columns:
        raise ValueError(f"Coluna não encontrada: {coluna_grupo}")

    resultados = []
    for valor, grupo in df.groupby(coluna_grupo):
        print(f"Resumindo {coluna_grupo} = {valor} ({len(grupo)} comentários)")
        resumo = resumir_dataframe(grupo, foco=f"{foco} | grupo {coluna_grupo}={valor}")
        resultados.append({
            "grupo": coluna_grupo,
            "valor": valor,
            "comentarios_analisados": len(grupo),
            "resumo_geral": resumo.get("resumo_geral", ""),
            "pontos_positivos": "; ".join(resumo.get("principais_pontos_positivos", [])),
            "reclamacoes": "; ".join(resumo.get("principais_reclamacoes", [])),
            "pontos_de_atencao": "; ".join(resumo.get("pontos_de_atencao", [])),
            "frase_executiva": resumo.get("frase_executiva", ""),
            "resumo_json": json.dumps(resumo, ensure_ascii=False),
        })
        time.sleep(0.5)

    saida = pd.DataFrame(resultados)
    nome_arquivo = f"resumos_por_{coluna_grupo.lower().replace(' ', '_')}.xlsx"
    saida.to_excel(nome_arquivo, index=False)
    files.download(nome_arquivo)
    return saida
